# Team T085 · Section E · Project 26
Phase 1: Machine Learning

The expanded run reuses the previously inspected test partition. Training-only selection and validation-only robustness are preserved; this is not a fresh independent holdout.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT=Path.cwd()
if ROOT.name=='notebooks': ROOT=ROOT.parent
assert (ROOT/'src').exists(), 'Start Jupyter in the project folder.'
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
TABLES=ROOT/'results/tables'
FIGURES=ROOT/'results/figures'
def table(name): return pd.read_csv(TABLES/name)
def figure(name): display(Image(filename=str(FIGURES/name)))


## Reusable preprocessing
Owner: Farhana. Inputs are V1-V28 and log1p(Amount). Time defines chronology. A full raw-frame pipeline fits StandardScaler and optional PCA exclusively on legitimate training rows.

In [2]:
from src.data import load_raw, load_split_indices
import yaml
cfg=yaml.safe_load((ROOT/'configs/config.yaml').read_text())
path=ROOT/'data/raw/creditcard.csv'
raw=load_raw(path)
parts=load_split_indices(raw,ROOT/'data/processed/split_indices.npz',path,cfg['split_fractions'])
from src.features import prepare,preprocess,engineering
normal=parts['train'][parts['train'].Class==0]
xt,pipeline=preprocess(normal)
mean_before=pipeline.named_steps['scaler'].mean_.copy()
xv,_=preprocess(parts['validation'],pipeline=pipeline)
np.testing.assert_array_equal(mean_before,pipeline.named_steps['scaler'].mean_)
np.testing.assert_allclose(mean_before,engineering(normal).mean(axis=0))
print('Training/validation shapes:',xt.shape,xv.shape)

Training/validation shapes: (170528, 29) (56958, 29)


In [3]:
from src.data import FEATURES
display(pd.DataFrame({'feature':FEATURES,'mean':mean_before,'scale':pipeline.named_steps['scaler'].scale_}))
for d in [5,10,20]:
    z,pp=preprocess(normal,pca_dimensions=d)
    print(d,'components; retained training variance:',pp.named_steps['pca'].explained_variance_ratio_.sum())
    assert z.shape==(len(normal),d)

,feature,mean,scale
0,V1,-0.157486,1.800936
1,V2,0.031547,1.586366
2,V3,0.510909,1.276925
3,V4,0.107344,1.348741
4,V5,-0.167235,1.300110
5,V6,0.061386,1.291372
6,V7,-0.066020,1.110073
7,V8,0.030706,1.174433
8,V9,0.024953,1.138973
9,V10,-0.015943,1.027344


5 components; retained training variance: 0.2494445863748564


10 components; retained training variance: 0.4428654055309252
20 components; retained training variance: 0.7882320340636083


## Leakage checklist
Reuse fitted transformations, never fit validation/test, preserve feature order, and never include Class in X. Study pipelines are saved locally in results/models; the Git repository documents regeneration. Provider PCA provenance remains unaudited.